# 📤 벡터 데이터 업로드

이 노트북에서는 제품 데이터를 임베딩 모델로 벡터화하여 Azure AI Search 인덱스에 업로드합니다.

## 📋 학습 목표

1. **임베딩 모델 연결하기** - Azure OpenAI text-embedding-3-large 모델 사용
2. **텍스트 벡터화하기** - 제품명과 리뷰를 3072차원 벡터로 변환
3. **배치 처리하기** - 대량 데이터를 효율적으로 임베딩
4. **벡터 업로드하기** - 생성된 벡터를 인덱스에 업데이트

## 🔍 주요 개념

### 임베딩(Embedding)이란?
- 텍스트를 고차원 벡터(숫자 배열)로 변환하는 과정
- 의미적으로 유사한 텍스트는 벡터 공간에서 가까운 위치에 배치
- 예: "노트북"과 "랩탑"의 벡터는 가까운 위치

### text-embedding-3-large
- OpenAI의 고성능 임베딩 모델
- **출력 차원**: 3072
- **최대 토큰**: 8191
- 빠른 속도와 우수한 성능

### 배치 처리
- API 호출 횟수 최소화를 위해 여러 텍스트를 한 번에 처리
- 비용 절감 및 속도 향상

---

## 1️⃣ 라이브러리 임포트

In [8]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI
from dotenv import load_dotenv
import pandas as pd
import os
import time
from tqdm import tqdm

print("✅ 라이브러리 임포트 완료!")

✅ 라이브러리 임포트 완료!


## 2️⃣ 환경 변수 로드 및 클라이언트 초기화

In [9]:
# 환경 변수 로드
load_dotenv(override=True)

# Azure AI Search 설정
search_endpoint = os.getenv("SEARCH_ENDPOINT")
search_key = os.getenv("SEARCH_ADMIN_KEY")
index_name = os.getenv("SEARCH_INDEX_NAME")

# Azure OpenAI 설정
openai_endpoint = os.getenv("AZURE_OPEN_AI_ENDPOINT")
openai_key = os.getenv("AZURE_OPEN_AI_KEY")
embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_CHAT_API_VERSION")

# SearchClient 초기화
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=index_name,
    credential=AzureKeyCredential(search_key)
)

# OpenAI Client 초기화
openai_client = AzureOpenAI(
    azure_endpoint=openai_endpoint,
    api_key=openai_key,
    api_version=api_version
)

print("✅ 클라이언트 초기화 완료")
print(f"   Search Endpoint: {search_endpoint}")
print(f"   OpenAI Endpoint: {openai_endpoint}")
print(f"   Embedding Model: {embedding_deployment}")
print(f"   Index: {index_name}")

✅ 클라이언트 초기화 완료
   Search Endpoint: https://ai-search-foundry-iq-cj.search.windows.net
   OpenAI Endpoint: https://foundry-aisearch-models.openai.azure.com
   Embedding Model: text-embedding-3-large
   Index: products-index


## 3️⃣ 데이터 로드

CSV 파일에서 제품 데이터를 로드합니다.

In [10]:
# CSV 파일 경로
csv_file = "../00-data/sample_data.csv"

# CSV 읽기
df = pd.read_csv(csv_file)

print(f"✅ CSV 파일 로드 완료")
print(f"   총 데이터 수: {len(df)}개")
print(f"   컬럼: {list(df.columns)}")

# 상위 3개 데이터 미리보기
print("\n📊 데이터 미리보기:")
df.head(3)

✅ CSV 파일 로드 완료
   총 데이터 수: 247개
   컬럼: ['id', 'title', 'brand', 'category', 'normal_price', 'image_link', 'review']

📊 데이터 미리보기:


,id,title,brand,category,normal_price,image_link,review
0,1,압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물),압소바,유아동,39000,https://foundryiq-image-gallery.azurewebsites....,신생아 백일 선물로 압소바6 레코딸랑이세트를 구매했는데 디자인이 귀여워서 선물 받는...
1,2,[압소바] 출산선물 티노딸랑이세트 (선물포장) (ATA367P1),압소바,유아동,40000,https://foundryiq-image-gallery.azurewebsites....,친구 아기 출산선물로 압소바 티노딸랑이세트를 구매했어요. 부드러운 소재라 아기 피부...
2,3,블루독베이비[hu] WH)블루독남아손수건SET 41170-006-01 손수건/타올/...,블루독베이비,유아동,22190,https://foundryiq-image-gallery.azurewebsites....,출산 준비하면서 블루독베이비 손수건 세트를 구입했는데 10장이나 들어 있어서 정말 ...


## 4️⃣ 임베딩 함수 정의

텍스트를 벡터로 변환하는 함수를 정의합니다.

### 전처리 전략
**text-embedding-3-large 모델 특성:**
- ✅ **특수문자 제거 불필요** - 모델이 자체 토큰화 수행
- ✅ **원시 텍스트 유지** - 맥락 보존이 더 중요
- ✅ **최소 전처리** - 공백 정리, 중복 제거 정도만 수행

### 배치 처리 전략
- 한 번에 최대 16개씩 처리
- API Rate Limit 고려
- 에러 발생 시 재시도 로직 포함

In [11]:
def preprocess_text(text):
    """
    임베딩을 위한 최소 전처리
    
    text-embedding-3-large은 원시 텍스트를 선호하므로
    과도한 전처리는 피하고 최소한의 정리만 수행
    """
    if not text or not isinstance(text, str):
        return "정보 없음"
    
    # 1. 기본 공백 정리
    text = text.strip()
    
    # 2. 연속된 공백을 하나로
    text = ' '.join(text.split())
    
    # 3. 연속된 줄바꿈 정리 (최대 2개까지만)
    while '\n\n\n' in text:
        text = text.replace('\n\n\n', '\n\n')
    
    # 4. 빈 문자열 처리
    if len(text) == 0:
        return "정보 없음"
    
    return text


def get_embeddings_batch(texts, model=embedding_deployment, max_retries=3):
    """
    텍스트 리스트를 벡터로 변환 (배치 처리)
    
    Args:
        texts: 텍스트 리스트
        model: 임베딩 모델명
        max_retries: 재시도 횟수
        
    Returns:
        벡터 리스트
    """
    for attempt in range(max_retries):
        try:
            # 최소 전처리 (공백 정리, 중복 제거)
            processed_texts = [preprocess_text(text) for text in texts]
            
            # 임베딩 생성
            response = openai_client.embeddings.create(
                input=processed_texts,
                model=model
            )
            
            # 결과 추출
            embeddings = [item.embedding for item in response.data]
            return embeddings
            
        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = (attempt + 1) * 2
                print(f"⚠️  에러 발생, {wait_time}초 후 재시도... ({attempt + 1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"❌ 임베딩 생성 실패: {str(e)}")
                # 실패 시 None 반환
                return [None] * len(texts)


# 전처리 함수 테스트
print("🧪 전처리 테스트:")
test_samples = [
    "  Samsung   Galaxy  S21  ",  # 과도한 공백
    "노트북\n\n\n\n좋아요",  # 과도한 줄바꿈
    "특수문자!@#$%^&*() 포함",  # 특수문자 (유지됨)
    "",  # 빈 문자열
    None  # None 값
]

for sample in test_samples:
    processed = preprocess_text(sample)
    print(f"   원본: {repr(sample)[:40]}")
    print(f"   처리: {repr(processed)}\n")

# 임베딩 함수 테스트
test_texts = ["테스트 제품", "샘플 리뷰", "특수문자!@# 포함"]
test_vectors = get_embeddings_batch(test_texts)
print(f"✅ 임베딩 함수 테스트 완료")
print(f"   입력: {len(test_texts)}개 텍스트")
print(f"   출력: {len(test_vectors)}개 벡터")
print(f"   벡터 차원: {len(test_vectors[0]) if test_vectors[0] else 0}")

# 벡터 원시 값 샘플 출력
if test_vectors[0]:
    print(f"\n🔢 벡터 원시 값 예시 (처음 10개 차원):")
    print(f"   텍스트: '{test_texts[0]}'")
    print(f"   벡터: {test_vectors[0][:20]}")
    print(f"   ... (총 3072개 숫자)")

print(f"\n💡 특수문자는 의도적으로 유지됩니다 (의미 보존)")


🧪 전처리 테스트:
   원본: '  Samsung   Galaxy  S21  '
   처리: 'Samsung Galaxy S21'

   원본: '노트북\n\n\n\n좋아요'
   처리: '노트북 좋아요'

   원본: '특수문자!@#$%^&*() 포함'
   처리: '특수문자!@#$%^&*() 포함'

   원본: ''
   처리: '정보 없음'

   원본: None
   처리: '정보 없음'

✅ 임베딩 함수 테스트 완료
   입력: 3개 텍스트
   출력: 3개 벡터
   벡터 차원: 3072

🔢 벡터 원시 값 예시 (처음 10개 차원):
   텍스트: '테스트 제품'
   벡터: [-0.01056711282581091, -0.01218600757420063, -0.009463687427341938, 0.04188184812664986, -0.0038438676856458187, 0.031040893867611885, -0.033956512808799744, 0.05496187135577202, 0.006628607865422964, -0.024098170921206474, -0.00379554252140224, 0.014924435876309872, 0.0028048758395016193, -0.018556881695985794, 0.0297200046479702, 0.017703134566545486, -0.008537453599274158, -0.009238169528543949, -0.039658889174461365, -0.011428912170231342]
   ... (총 3072개 숫자)

💡 특수문자는 의도적으로 유지됩니다 (의미 보존)


## 5️⃣ 전체 데이터 벡터화

모든 제품의 content와 review를 벡터로 변환합니다.

### Content Vector 구성

**마크다운 형식**으로 구조화하여 임베딩 모델이 각 필드를 명확히 이해:⏱️ **예상 소요 시간**: 약 2-5분 (247개 제품 기준)

```

**제품명**: [title]
**브랜드**: [brand]
**카테고리**: [category]

```

In [13]:
def create_content_text(row):
    """
    제품 정보를 마크다운 형식으로 구조화
    
    Args:
        row: DataFrame row (title, brand, category)
        
    Returns:
        마크다운 형식의 구조화된 텍스트
    """
    content_parts = []
    
    # 제품명
    if pd.notna(row['title']) and str(row['title']).strip():
        content_parts.append(f"**제품명**: {row['title']}")
    
    # 브랜드
    if pd.notna(row['brand']) and str(row['brand']).strip():
        content_parts.append(f"**브랜드**: {row['brand']}")
    
    # 카테고리
    if pd.notna(row['category']) and str(row['category']).strip():
        content_parts.append(f"**카테고리**: {row['category']}")
    
    # 줄바꿈으로 결합
    return "\n".join(content_parts) if content_parts else "정보 없음"


# 배치 크기 설정
BATCH_SIZE = 16

print(f"🚀 벡터화 시작: {len(df)}개 제품")
print(f"   배치 크기: {BATCH_SIZE}")
print(f"   예상 API 호출 횟수: {(len(df) * 2 + BATCH_SIZE - 1) // BATCH_SIZE}회\n")

# Content 텍스트 생성 (title + brand + category를 마크다운 형식으로)
print("📝 Content 텍스트 생성 중...")
df['content_text'] = df.apply(create_content_text, axis=1)

# 샘플 확인
print("\n📄 Content 텍스트 샘플:")
print("-" * 60)
print(df['content_text'].iloc[0])
print("-" * 60)
print()

# content 벡터화
print("📝 Content 벡터화 중...")
content_vectors = []
for i in tqdm(range(0, len(df), BATCH_SIZE), desc="Content"):
    batch = df['content_text'].iloc[i:i+BATCH_SIZE].tolist()
    vectors = get_embeddings_batch(batch)
    content_vectors.extend(vectors)
    time.sleep(0.5)  # Rate limit 방지

df['content_vector'] = content_vectors

# review 벡터화
print("\n💬 리뷰(review) 벡터화 중...")
review_vectors = []
for i in tqdm(range(0, len(df), BATCH_SIZE), desc="Review"):
    batch = df['review'].iloc[i:i+BATCH_SIZE].tolist()
    vectors = get_embeddings_batch(batch)
    review_vectors.extend(vectors)
    time.sleep(0.5)  # Rate limit 방지

df['review_vector'] = review_vectors

# 결과 확인
success_count = df['content_vector'].notna().sum()
print(f"\n✅ 벡터화 완료!")
print(f"   성공: {success_count}/{len(df)}개")
print(f"   실패: {len(df) - success_count}개")

# 샘플 확인
if success_count > 0:
    sample = df[df['content_vector'].notna()].iloc[0]
    print(f"\n📊 샘플 벡터 정보:")
    print(f"   제품명: {sample['title'][:50]}...")
    print(f"   Content 원본 텍스트:")
    print(f"      {sample['content_text'][:100]}...")
    print(f"   Content Vector 차원: {len(sample['content_vector']) if sample['content_vector'] else 0}")
    print(f"   Review Vector 차원: {len(sample['review_vector']) if sample['review_vector'] else 0}")


🚀 벡터화 시작: 247개 제품
   배치 크기: 16
   예상 API 호출 횟수: 31회

📝 Content 텍스트 생성 중...

📄 Content 텍스트 샘플:
------------------------------------------------------------
**제품명**: 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)
**브랜드**: 압소바
**카테고리**: 유아동
------------------------------------------------------------

📝 Content 벡터화 중...


Content: 100%|██████████| 16/16 [00:17<00:00,  1.11s/it]



💬 리뷰(review) 벡터화 중...


Review: 100%|██████████| 16/16 [00:23<00:00,  1.44s/it]


✅ 벡터화 완료!
   성공: 247/247개
   실패: 0개

📊 샘플 벡터 정보:
   제품명: 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)...
   Content 원본 텍스트:
      **제품명**: 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)
**브랜드**: 압소바
**카테고리**: 유아동...
   Content Vector 차원: 3072
   Review Vector 차원: 3072


## 6️⃣ Azure AI Search에 벡터 업로드

생성된 벡터를 인덱스에 업데이트합니다.

### merge_or_upload 방식
- **기존 문서**: 벡터 필드만 업데이트 (다른 필드 유지)
- **신규 문서**: 새로 추가

In [14]:
# 업로드용 문서 준비
documents = []

for _, row in df.iterrows():
    # None이 아닌 경우만 업로드
    if row['content_vector'] is not None and row['review_vector'] is not None:
        doc = {
            "id": str(row['id']),
            "content_vector": row['content_vector'],
            "review_vector": row['review_vector']
        }
        documents.append(doc)

print(f"📤 업로드 준비: {len(documents)}개 문서")
print(f"   업로드할 필드: id, content_vector, review_vector\n")

# 배치 업로드 (1000개씩)
UPLOAD_BATCH_SIZE = 1000
total_uploaded = 0
failed = 0

for i in tqdm(range(0, len(documents), UPLOAD_BATCH_SIZE), desc="Upload"):
    batch = documents[i:i+UPLOAD_BATCH_SIZE]
    
    try:
        result = search_client.merge_or_upload_documents(documents=batch)
        
        # 결과 확인
        for item in result:
            if item.succeeded:
                total_uploaded += 1
            else:
                failed += 1
                print(f"⚠️  업로드 실패: ID={item.key}, Error={item.error_message}")
        
    except Exception as e:
        print(f"❌ 배치 업로드 실패: {str(e)}")
        failed += len(batch)

print(f"\n{'='*80}")
print(f"✅ 업로드 완료!")
print(f"{'='*80}")
print(f"   성공: {total_uploaded}개")
print(f"   실패: {failed}개")
print(f"   총 문서: {len(documents)}개")

📤 업로드 준비: 247개 문서
   업로드할 필드: id, content_vector, review_vector



Upload: 100%|██████████| 1/1 [00:06<00:00,  6.86s/it]


✅ 업로드 완료!
   성공: 247개
   실패: 0개
   총 문서: 247개


## 7️⃣ 업로드 결과 검증

벡터가 정상적으로 업로드되었는지 확인합니다.

In [17]:
# 샘플 문서 조회 (벡터 필드는 Retrievable=False이므로 직접 조회 불가)
sample_id = str(df.iloc[0]['id'])

try:
    # 기본 필드만 조회
    doc = search_client.get_document(
        key=sample_id,
        selected_fields=["id", "title", "brand", "category"]
    )
    
    print(f"✅ 문서 조회 성공!")
    print(f"\n📋 문서 ID: {doc['id']}")
    print(f"   제품명: {doc.get('title', 'N/A')}")
    print(f"   브랜드: {doc.get('brand', 'N/A')}")
    print(f"   카테고리: {doc.get('category', 'N/A')}")
    
    # 벡터 필드는 Retrievable=False → 벡터 검색으로 존재 여부 확인
    from azure.search.documents.models import VectorizedQuery
    
    sample_vector = df.iloc[0]['content_vector']
    if sample_vector:
        results = search_client.search(
            search_text=None,
            vector_queries=[
                VectorizedQuery(
                    vector=sample_vector,
                    k_nearest_neighbors=1,
                    fields="content_vector"
                )
            ],
            top=1,
            select=["id", "title"]
        )
        result_list = list(results)
        if result_list:
            print(f"\n🎯 벡터 검색 검증:")
            print(f"   content_vector: ✅ 벡터 검색 정상 동작")
            print(f"   매칭된 문서: {result_list[0]['title'][:50]}")
            print(f"   점수: {result_list[0]['@search.score']:.4f}")
        else:
            print(f"\n❌ content_vector 벡터 검색 결과 없음")
    
    print(f"\n💡 참고: 벡터 필드는 Retrievable=False이므로")
    print(f"   get_document()로 벡터 값 자체는 조회할 수 없습니다.")
    print(f"   벡터 검색(vector search)으로만 활용됩니다.")
    
except Exception as e:
    print(f"❌ 문서 조회 실패: {str(e)}")

✅ 문서 조회 성공!

📋 문서 ID: 1
   제품명: 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)
   브랜드: 압소바
   카테고리: 유아동

🎯 벡터 검색 검증:
   content_vector: ✅ 벡터 검색 정상 동작
   매칭된 문서: 압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)
   점수: 1.0000

💡 참고: 벡터 필드는 Retrievable=False이므로
   get_document()로 벡터 값 자체는 조회할 수 없습니다.
   벡터 검색(vector search)으로만 활용됩니다.


---

## ✅ 완료!

벡터 데이터가 성공적으로 업로드되었습니다.

### 📝 작업 요약

1. ✅ **247개 제품 정보** → content_vector (3072차원)
   - 제품명 + 브랜드 + 카테고리 (마크다운 형식)
2. ✅ **247개 리뷰** → review_vector (3072차원)
3. ✅ Azure AI Search 인덱스에 업로드
4. ✅ 업로드 결과 검증 완료

### 🚀 다음 단계

**03-vector_search.ipynb**에서 다음 작업을 수행합니다:
1. 벡터 검색 실행 (의미 기반 검색)
2. 키워드 검색과 비교
3. 하이브리드 검색 (키워드 + 벡터)

### 💡 참고

- 벡터 검색은 **의미적 유사도** 기반
- "노트북" 검색 시 "랩탑", "컴퓨터" 같은 유사어도 검색
- 키워드 검색보다 더 자연스러운 검색 결과 제공

---

## 🧭 다음 단계

| ⬅️ 이전 | 🏠 목차 | ➡️ 다음 |
|:---------|:-------:|--------:|
| [Lab 03-1: 벡터 인덱스 업데이트](01-update_index.ipynb) | [워크숍 홈](../README.md) | [Lab 03-3: 벡터 검색](03-vector_search.ipynb) |